In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Defining the path to PDF test document
FILE_PATH = "./data/sample.pdf"

def load_and_chunk_document(file_path: str):
    """
    Loads a PDF document and splits it into overlapping chunks for embedding.
    """
    print(f"Loading document from: {file_path}")
    
    # Initialize the loader
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    
    print(f"Successfully loaded {len(documents)} pages.")

    # 2. Configure the Text Splitter
    # We use RecursiveCharacterTextSplitter because it tries to split on paragraphs first, 
    # then sentences, then words, keeping related text together as much as possible.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,       # The max characters per chunk
        chunk_overlap=200,     # The safety net overlap
        length_function=len,   # How we measure length
        add_start_index=True   # Injects metadata about where the chunk came from
    )
    
    # 3. Execute the splitting
    chunks = text_splitter.split_documents(documents)
    
    print(f"Document split into {len(chunks)} individual chunks.")
    return chunks

# Execute our function
document_chunks = load_and_chunk_document(FILE_PATH)

# Inspecting a single chunk to verify if it works(the logic) or not
if document_chunks:
    print("\n--- SAMPLE CHUNK [Index 5] ---")
    print(document_chunks[5].page_content)
    print("\n--- CHUNK METADATA ---")
    print(document_chunks[5].metadata)

Loading document from: ./data/sample.pdf
Successfully loaded 12 pages.
Document split into 87 individual chunks.

--- SAMPLE CHUNK [Index 5] ---
26 IEEE Spectrum |September 2005 |NA www.spectrum.ieee.org
COMPUTERWORLD
failure. Among them: poorly defined and slowly
evolving design requirements; overly ambitious
schedules; and the lack of a plan to guide hard-
ware purchases, network deployments, and soft-
ware development for the bureau.
Fine concluded that four years after terrorists
crashed jetliners into the World Trade Center and
the Pentagon, the FBI, which had been criticized
for not “connecting the dots” in time to prevent
the attacks, still did not have the software nec-
essary to connect any new dots that might come
along. And won’t for years to come. 
“The archaic Automated Case Support sys-
tem—which some agents have avoided using—
iscumbersome, inefficient, and limited in its capa-
bilities, and does not manage, link, research,
analyze, and share information as effectively o

In [5]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# ==========================================
# STAGE 1: SECURITY & SETUP
# ==========================================
load_dotenv()
if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("CRITICAL ERROR: GOOGLE_API_KEY not found in .env file.")
else:
    print("Security Check: API Key loaded successfully.")

# ==========================================
# STAGE 2: DATA INGESTION & CHUNKING
# ==========================================
FILE_PATH = "./data/sample.pdf"
print(f"\n[Stage 2] Loading document from: {FILE_PATH}")

loader = PyPDFLoader(FILE_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
document_chunks = text_splitter.split_documents(documents)
print(f"Document securely split into {len(document_chunks)} chunks.")

# ==========================================
# STAGE 3: EMBEDDING & VECTOR STORAGE
# ==========================================
print("\n[Stage 3] Initializing Google gemini-embedding-001 model...")

# THE FIX: We updated the model name to the current 2026 standard
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

CHROMA_PATH = "./chroma_db"
print("Connecting to Google Servers for vector calculation. Please wait...")

# Generate embeddings and save to disk
vector_db = Chroma.from_documents(
    documents=document_chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"\n[SUCCESS] Pipeline Complete! Vector database permanently saved to: {CHROMA_PATH}")

Security Check: API Key loaded successfully.

[Stage 2] Loading document from: ./data/sample.pdf
Document securely split into 87 chunks.

[Stage 3] Initializing Google gemini-embedding-001 model...
Connecting to Google Servers for vector calculation. Please wait...

[SUCCESS] Pipeline Complete! Vector database permanently saved to: ./chroma_db
